In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import wandb
import unsloth
from datasets import Dataset
from transformers import AutoTokenizer
from finetune_mbert import train_eval_model
from experiment import preprocess_text

os.environ["CUDA_VISIBLE_DEVICES"]="0"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
def preprocess(example):
    return {'text': preprocess_text(example['text']), 'labels': example['labels']}

def load_emotionality_dataset(tokenizer, label_bounds = {0: 'present'}, **tokenizer_args):
    DATA_PATH = Path("../data/2026-05-21/corpus_emotions.json")
    data = pd.read_json(DATA_PATH)
    labels = pd.cut(
        data['emotionality'], 
        bins=list(label_bounds.keys()) + [np.max(data['emotionality'])+1],
        labels=list(label_bounds.values()),
        right=False
    ).astype('object')
    data = Dataset.from_dict({'text': data['content'], 'labels': labels})
    data = data.map(preprocess)
    labels_map = {label: ind for ind, label in enumerate(set(data['labels']))}

    def tokenize(example):
        labels = list(map(lambda m: labels_map.get(m), example['labels']))
        example = tokenizer(example["text"], padding="max_length", truncation=True, **tokenizer_args)
        example['labels'] = labels
        return example

    final_dst = data.map(tokenize, batched=True, remove_columns=['text'])
    final_dst.set_format(type="torch")
    return final_dst, labels_map

## ModernBert

In [3]:
tokenizer_args = {
  "return_token_type_ids": "false",
  "max_length": 512
}
models_config = [
    ("VSSA-SDSA/LT-MLKM-modernBERT", "outputs/emotion-classifier-modernbert-unsloth")
]
label_config = [
    {0: 'neutral', 0.5: 'present'}, 
    {0: 'neutral', 0.5: 'medium', 1.5: 'high'},
    {0: 'neutral', 0.5: 'moderate', 1.0: 'high', 1.5: 'strong', 2.0: 'extreme'}
]

In [4]:
for model_name, output_dir in models_config:
    for ind, label_bounds in enumerate(label_config):
        print(f"Training model for emotion detection with model {model_name} using discretization to {len(label_bounds)} classes")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        ds, labels_map = load_emotionality_dataset(tokenizer, label_bounds=label_bounds, **tokenizer_args)
        run = wandb.init(project="emotion-detection-lt-unsloth-8bit-lora", name=f"{model_name}-{len(label_bounds)}class-{ind}")
        train_eval_model(
            model_name=model_name,
            dataset=ds,
            labels_map=labels_map,
            use_lora=False,
            output_dir=f"{output_dir}-{ind}", 
            batch_size=2, 
            num_epochs=20, 
            eval_batch_size=16,
            report_to=['tensorboard', 'wandb']
        )

Training model for emotion detection with model VSSA-SDSA/LT-MLKM-modernBERT using discretization to 2 classes


Map:   0%|          | 0/2660 [00:00<?, ? examples/s]

Map:   0%|          | 0/2660 [00:00<?, ? examples/s]

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/paudan/.netrc.
wandb: Currently logged in as: danpaulius (danpaulius-ktu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in outputs/emotion-classifier-modernbert-unsloth-2/checkpoint-3728/model.safetensors
The following columns in the Evaluation set don't have a corresponding argument in `ModernBertForSequenceClassification.forward` and have been ignored: token_type_ids. If token_type_ids are not expected by `ModernBertForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 399
  Batch size = 32
[tensorboardX.x2num|WARNING]NaN or Inf found in input tensor.
Saving model checkpoint to outputs/emotion-classifier-modernbert-unsloth-2/checkpoint-4194
Configuration saved in outputs/emotion-classifier-modernbert-unsloth-2/checkpoint-4194/config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in outputs/emotion-classifier-modernbert-unsloth-2/checkpoint-4194/model.safetensors


Training completed. Do not forget to share your model on huggingface.co/models =)


Loading best model from outputs/emotion-classifier-modernbert-unsloth-2/checkpoint-1864 (score: 0.41102756892230574).
The following columns in the test set don't have a corresponding argument in `ModernBertForSequenceClassification.forward` and have been ignored: token_type_ids. If token_type_ids are not expected by `ModernBertForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 400
  Batch size = 32


              precision    recall  f1-score   support

    moderate       0.36      0.39      0.37       119
      strong       0.20      0.23      0.21        13
     extreme       0.20      0.10      0.13        21
        high       0.45      0.42      0.44       132
     neutral       0.50      0.53      0.51       115

    accuracy                           0.42       400
   macro avg       0.34      0.33      0.33       400
weighted avg       0.42      0.42      0.42       400

